In [1]:
from tvDatafeed import TvDatafeed, Interval
import time
import pandas as pd
from tradingview_screener import Query, col

In [5]:
tv = TvDatafeed()

# Define your stock and exchange
symbol = "OIH"
exchange = "EGX"

# n_bars=250 gives you approx. 1 year of trading days (excluding weekends/holidays)
data = tv.get_hist(symbol=symbol, exchange=exchange, interval=Interval.in_daily, n_bars=250)

data

,symbol,open,high,low,close,volume
datetime,,,,,,
2025-03-06 10:00:00,EGX:OIH,0.525,0.536,0.522,0.533,47113899.0
2025-03-09 10:00:00,EGX:OIH,0.533,0.545,0.533,0.540,51199767.0
2025-03-10 10:00:00,EGX:OIH,0.540,0.550,0.530,0.531,74719918.0
2025-03-11 10:00:00,EGX:OIH,0.531,0.531,0.525,0.527,28226301.0
2025-03-12 10:00:00,EGX:OIH,0.527,0.532,0.526,0.526,17713581.0
...,...,...,...,...,...,...
2026-03-12 10:00:00,EGX:OIH,1.230,1.230,1.210,1.220,26214407.0
2026-03-15 10:00:00,EGX:OIH,1.220,1.240,1.210,1.220,47247331.0
2026-03-16 10:00:00,EGX:OIH,1.220,1.230,1.210,1.230,31429067.0


In [48]:
# 1. Correct way to filter for Egypt
print("Searching for all EGX symbols...")
rows, df_symbols = (Query()
    .set_markets('egypt')  # This is the key fix
    .select('name', 'description', 'close', 'change', 'volume', 'type')
    .limit(500)
    .get_scanner_data())

all_tickers = df_symbols['name'].tolist()
print(f"✅ Success! Found {len(all_tickers)} symbols in Egypt.")

# 2. Initialize Datafeed
tv = TvDatafeed()
tv
# # 3. Download the first 5 as a test (to avoid long wait times)
# for ticker in all_tickers[:5]: 
#     try:
#         # Note: In tvDatafeed, some Egyptian stocks need 'EGX' as the exchange
#         data = tv.get_hist(symbol=ticker, exchange='EGX', interval=Interval.in_daily, n_bars=250)
        
#         if data is not None:
#             print(f"📊 {ticker}: Downloaded {len(data)} rows.")
#         else:
#             print(f"⚠️ {ticker}: No data found on EGX exchange.")
            
#         time.sleep(0.5) 
#     except Exception as e:
#         print(f"❌ Error on {ticker}: {e}")

Searching for all EGX symbols...
✅ Success! Found 253 symbols in Egypt.


In [50]:
all_tickers

pd.DataFrame({"ticker": all_tickers}).to_csv("EGX.csv", index=False)

In [55]:
df_latest = df_symbols[['name', 'description', 'close', 'change', 'volume']]
df_latest.columns = ['Ticker', 'Name', 'Price_EGP', 'Change_%', 'Volume']

df_sorted = df_latest.sort_values(by='Volume', ascending=False)

pd.DataFrame(df_sorted).to_csv("EGXDetails.csv", index=False)

# df_sorted